# Day 050 — Exercise 4: narrate_insights

**What you'll build:** `narrate_insights(eda, model_report, title, model='llama3.2') -> str` — use Ollama to generate a 3-sentence executive summary from the EDA dict and model report dict. Include a try/except fallback for when Ollama is not running.

**Why it matters:** This is the AI layer that transforms structured statistics into plain-English insight. A stakeholder reading the output should understand what the data contains, what the main pattern is, and what action to take — without seeing a single number.

## Provided: Setup + load_and_clean + run_eda + train_and_evaluate

In [ ]:
import io
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import ollama
warnings.filterwarnings('ignore')


def make_sample_data(n: int = 300, seed: int = 42) -> pd.DataFrame:
    """
    Retail sales dataset.
    Columns: date (str), region, category, units_sold, price, discount, revenue.
    Revenue = units_sold*4 + price*1.5 - discount*150 + noise (5% nulls injected).
    """
    rng      = np.random.default_rng(seed)
    dates    = pd.date_range('2023-01-01', periods=n, freq='D').strftime('%Y-%m-%d')
    region   = rng.choice(['North', 'South', 'East', 'West'], n)
    category = rng.choice(['Electronics', 'Clothing', 'Food', 'Books'], n)
    units    = rng.integers(1, 50, n)
    price    = rng.uniform(5.0, 200.0, n).round(2)
    discount = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], n)
    revenue  = (units * 4.0 + price * 1.5 - discount * 150
                + rng.standard_normal(n) * 20).round(2)
    null_idx = rng.choice(n, size=max(1, int(n * 0.05)), replace=False)
    revenue  = revenue.astype(float)
    revenue[null_idx] = np.nan
    return pd.DataFrame({
        'date':       pd.Series(dates),
        'region':     region,
        'category':   category,
        'units_sold': units,
        'price':      price,
        'discount':   discount,
        'revenue':    revenue,
    })


def load_and_clean(source) -> pd.DataFrame:
    """
    Load from CSV string / file path / DataFrame and clean.

    Steps applied in order:
      1. Parse source into a DataFrame
      2. Detect and parse date/time columns to datetime64
      3. Fill numeric NaN with column median
      4. Drop exact duplicate rows
    """
    if isinstance(source, pd.DataFrame):
        df = source.copy()
    elif isinstance(source, str) and ('\n' in source or ',' in source[:200]):
        df = pd.read_csv(io.StringIO(source))
    else:
        df = pd.read_csv(source)

    # Detect date columns by name
    for col in df.columns:
        if any(kw in col.lower() for kw in ('date', 'time', 'created', 'updated')):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
            except Exception:
                pass

    # Fill numeric NaN with column median
    for col in df.select_dtypes(include='number').columns:
        median = df[col].median()
        df[col] = df[col].fillna(median)

    # Drop duplicates
    df = df.drop_duplicates().reset_index(drop=True)
    return df


def run_eda(df: pd.DataFrame) -> dict:
    """
    Compute an EDA summary dict with keys:
        shape, columns, dtypes, null_counts,
        numeric_summary, correlations, category_counts,
        numeric_cols, cat_cols
    """
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cat_cols = df.select_dtypes(include='object').columns.tolist()

    numeric_summary = {}
    for col in num_cols:
        s = df[col].dropna()
        numeric_summary[col] = {
            'mean':   round(float(s.mean()),   4),
            'std':    round(float(s.std()),    4),
            'min':    round(float(s.min()),    4),
            'max':    round(float(s.max()),    4),
            'median': round(float(s.median()), 4),
        }

    correlations = {}
    if len(num_cols) >= 2:
        cm = df[num_cols].corr()
        for col in num_cols:
            correlations[col] = {
                other: round(float(cm.loc[col, other]), 4)
                for other in num_cols if other != col
            }

    category_counts = {
        col: df[col].value_counts().head(10).to_dict()
        for col in cat_cols
    }

    return {
        'shape':           {'rows': int(df.shape[0]), 'cols': int(df.shape[1])},
        'columns':         df.columns.tolist(),
        'dtypes':          {c: str(t) for c, t in df.dtypes.items()},
        'null_counts':     df.isnull().sum().to_dict(),
        'numeric_summary': numeric_summary,
        'correlations':    correlations,
        'category_counts': category_counts,
        'numeric_cols':    num_cols,
        'cat_cols':        cat_cols,
    }


def train_and_evaluate(df: pd.DataFrame, target_col: str,
                        test_size: float = 0.2,
                        random_state: int = 42) -> dict:
    """
    Auto-select numeric features, scale, train LinearRegression with 5-fold CV,
    and evaluate on a held-out test set.

    Returns dict with keys:
        target, features, cv_r2 (mean/std), test_r2, test_rmse, test_mae,
        coefficients (feature → value), n_train, n_test
    """
    if target_col not in df.columns:
        return {'error': f'target column {target_col!r} not found'}

    num_cols     = df.select_dtypes(include='number').columns.tolist()
    feature_cols = [c for c in num_cols if c != target_col]

    if not feature_cols:
        return {'error': 'no numeric feature columns found'}

    sub   = df[feature_cols + [target_col]].dropna()
    X     = sub[feature_cols]
    y     = sub[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    scaler   = StandardScaler()
    X_tr_s   = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_cols)
    X_te_s   = pd.DataFrame(scaler.transform(X_test),      columns=feature_cols)

    model = LinearRegression()
    kf    = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_tr_s, y_train, cv=kf, scoring='r2')

    model.fit(X_tr_s, y_train)
    y_pred = model.predict(X_te_s)

    return {
        'target':    target_col,
        'features':  feature_cols,
        'cv_r2':     {'mean': round(float(cv_scores.mean()), 4),
                      'std':  round(float(cv_scores.std()),  4)},
        'test_r2':   round(float(r2_score(y_test, y_pred)),                         4),
        'test_rmse': round(float(np.sqrt(mean_squared_error(y_test, y_pred))),       2),
        'test_mae':  round(float(mean_absolute_error(y_test, y_pred)),               2),
        'coefficients': {
            col: round(float(c), 4)
            for col, c in zip(feature_cols, model.coef_)
        },
        'n_train': int(len(X_train)),
        'n_test':  int(len(X_test)),
    }

## Your Implementation

In [ ]:
def narrate_insights(eda: dict, model_report: dict,
                     title: str = 'Dataset',
                     model: str = 'llama3.2') -> str:
    """
    Build a context string from eda + model_report, then call
    ollama.chat to produce a 3-sentence executive summary.

    If Ollama is unavailable, return a plain-text fallback summary.
    """
    shape    = eda.get('shape', {})
    num_cols = eda.get('numeric_cols', [])
    cat_cols = eda.get('cat_cols', [])

    # TODO: Build context string from EDA + model_report
    # context = ...

    # TODO: Build a user prompt asking for a 3-sentence executive summary
    # prompt = ...

    # TODO: Call ollama.chat(model=model, messages=[{'role':'user','content':prompt}])
    # Return response['message']['content'].strip()
    # Wrap in try/except — return a fallback string if Ollama is unavailable

    try:
        # TODO: response = ollama.chat(model=model, messages=[...])
        # TODO: return response['message']['content'].strip()
        pass
    except Exception:
        # Fallback: return a structured plain-text summary
        lines = [f'Analysis of {title}: '
                 f"{shape.get('rows', '?')} rows, {shape.get('cols', '?')} columns."]
        if num_cols:
            lines.append(f'Numeric features: {", ".join(num_cols)}.')
        if model_report and 'test_r2' in model_report:
            lines.append(
                f"Predictive model (LinearRegression \u2192 {model_report['target']}): "
                f"Test R\u00b2={model_report['test_r2']:.4f}, "
                f"RMSE={model_report['test_rmse']:.2f}."
            )
        return ' '.join(lines)

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    df    = load_and_clean(make_sample_data(200))
    eda   = run_eda(df)
    mr    = train_and_evaluate(df, 'revenue')

    # Check 1: returns a string
    try:
        result = narrate_insights(eda, mr, title='Retail Sales')
        assert isinstance(result, str), \
            f'expected str, got {type(result).__name__}'
        passed += 1; print(f'\u2705 Check 1: narrate_insights returns str')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: string is non-empty (> 20 chars)
    try:
        assert len(result) > 20, \
            f'narrative is too short ({len(result)} chars)'
        passed += 1; print(f'\u2705 Check 2: narrative length={len(result)} > 20')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: works with empty model_report (no model run)
    try:
        fallback = narrate_insights(eda, {}, title='Test')
        assert isinstance(fallback, str) and len(fallback) > 10
        passed += 1; print(f'\u2705 Check 3: works with empty model_report')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: does not raise on minimal EDA dict
    try:
        minimal_eda = {'shape': {'rows': 10, 'cols': 3},
                       'numeric_cols': ['a'], 'cat_cols': [],
                       'numeric_summary': {}, 'correlations': {},
                       'category_counts': {}}
        out = narrate_insights(minimal_eda, {}, title='Min')
        assert isinstance(out, str)
        passed += 1; print(f'\u2705 Check 4: handles minimal EDA dict')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: narrative printed for inspection
    try:
        print(f'\n--- Narrative ({len(result)} chars) ---')
        print(result[:500])
        passed += 1; print(f'\n\u2705 Check 5: narrative displayed')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def narrate_insights(eda: dict, model_report: dict,
                     title: str = 'Dataset',
                     model: str = 'llama3.2') -> str:
    """
    Generate a 3-sentence executive summary via Ollama (llama3.2).
    Falls back to a structured plain-text summary if Ollama is unavailable.
    """
    shape    = eda.get('shape', {})
    num_cols = eda.get('numeric_cols', [])
    cat_cols = eda.get('cat_cols', [])

    context_parts = [
        f"Dataset: {title}",
        f"Shape: {shape.get('rows', '?')} rows \u00d7 {shape.get('cols', '?')} columns",
        f"Numeric columns: {', '.join(num_cols) if num_cols else 'none'}",
        f"Categorical columns: {', '.join(cat_cols) if cat_cols else 'none'}",
    ]

    num_summary = eda.get('numeric_summary', {})
    for col, stats in list(num_summary.items())[:4]:
        context_parts.append(
            f"  {col}: mean={stats['mean']:.2f}, std={stats['std']:.2f}, "
            f"min={stats['min']:.2f}, max={stats['max']:.2f}"
        )

    if model_report and 'test_r2' in model_report:
        context_parts.append(
            f"LinearRegression predicting {model_report['target']}: "
            f"CV R\u00b2={model_report['cv_r2']['mean']:.4f} \u00b1 {model_report['cv_r2']['std']:.4f}, "
            f"Test R\u00b2={model_report['test_r2']:.4f}, "
            f"RMSE={model_report['test_rmse']:.2f}"
        )
        top = sorted(model_report.get('coefficients', {}).items(),
                     key=lambda kv: abs(kv[1]), reverse=True)
        if top:
            context_parts.append(
                f"Strongest predictor: {top[0][0]} (coef={top[0][1]:.4f})"
            )

    context = '\n'.join(context_parts)
    prompt  = (
        f"You are a concise data analyst. Write a 3-sentence executive summary "
        f"for a non-technical stakeholder based on this analysis:\n\n{context}\n\n"
        f"Cover: (1) what the data contains, (2) the key pattern or insight, "
        f"(3) one concrete recommendation."
    )

    try:
        response = ollama.chat(
            model=model,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return response['message']['content'].strip()
    except Exception:
        lines = [f"Analysis of {title}: {shape.get('rows', '?')} rows, "
                 f"{shape.get('cols', '?')} columns."]
        if num_cols:
            lines.append(f"Numeric features: {', '.join(num_cols)}.")
        if model_report and 'test_r2' in model_report:
            lines.append(
                f"Predictive model (LinearRegression \u2192 {model_report['target']}): "
                f"Test R\u00b2={model_report['test_r2']:.4f}, "
                f"RMSE={model_report['test_rmse']:.2f}."
            )
        return ' '.join(lines)
```

</details>